In [1]:
from google import genai
from google.genai import types
from PIL import Image

In [2]:
# 1. Initialize the client (automatically picks up GEMINI_API_KEY from environment)
client = genai.Client(
    vertexai=True,
    project="infinitas-ds-ai-poc",
    location="us-central1",
)

In [3]:
# 2. Open your local image file

path_batch1 = ["test_images/First_Tamper.png", "test_images/first-tamper.png", "test_images/pink.png","test_images/combined.png",
         "test_images/neat.png", "test_images/neatest.png", "test_images/cropped-obvious.png"] # explore strength and weaknesses

path_batch2 = ["test_images/payslip1.jpeg", "test_images/payslip10.jpeg", "test_images/payslip2.png",
               "test_images/payslip20.png","test_images/payslip3.png", "test_images/payslip30.png"] # explore payslip with white space but different fonts

paths = path_batch2
images = []

for path in paths:
    images.append(Image.open(path))

print(len(images))

6


In [4]:
# Read the V1 prompt

with open("prompt-library/V1.txt", "r", encoding="utf-8") as file:
    file_content = file.read()

prompt_v1 = file_content
print(type(prompt_v1))

# Read the V1 prompt split into system instruction + task prompt

with open("prompt-library/V1_system.txt", "r", encoding="utf-8") as file:
    system_prompt_v1 = file.read()

with open("prompt-library/V1_task.txt", "r", encoding="utf-8") as file:
    task_prompt_v1 = file.read()

<class 'str'>


## Result logging

Every model call below is logged to `notebook_results/results_log.jsonl` via `result_logger.log_result()`.
This keeps a permanent record across batches without cluttering this notebook.

**For each new batch of images:**
1. Bump `BATCH_ID` below (e.g. `"batch2"`)
2. Update `paths` in the image-loading cell to point at the new images
3. Re-run the experiment cells as usual — results from previous batches stay in the log

Use the "Review logged results" section at the end to compare across batches.

In [5]:
import time
from result_logger import log_result, load_results

BATCH_ID = "batch2"  # bump this for each new set of test images

## Try with the same model for now to test the workflow
### Models (WIP)
- "gemini-2.5-flash"
- "gemini-3.5-flash"
- "gemini-3.5-pro"

In [6]:
image_list = list(zip(paths, images))  

for image_path, image in image_list:

    start = time.perf_counter()

    # 3. Generate content by passing both the image object and your text prompt
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            prompt_v1,
            image
        ], config = types.GenerateContentConfig(
            temperature = 0.05  # To change to variable this later
        )
    )

    latency_s = round(time.perf_counter() - start, 2)

    # 4. Print the text result
    print(response.text)

    log_result(
        batch_id=BATCH_ID,
        image_path=image_path,
        prompt_id="V1",
        model="gemini-2.5-flash",
        temperature=0.05,
        raw_response=response.text,
        latency_s=latency_s,
    )

```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document displays a consistent typographic style throughout. All text elements, including headers, labels, and numerical values (both bolded and non-bolded), exhibit uniform font weight, sharpness, and resolution. There are no discernible inconsistencies in anti-aliasing, pixelation, or rendering quality that would suggest digital alteration after recapture. The background behind the text is uniform, and no color or contrast discontinuities are observed around any characters or numbers. All text appears to be part of the original digital generation of the document.",
  "signals_detected": [],
  "notes": "The document is a synthetic, digitally generated image, and its visual consistency strongly supports its authenticity as a generated image without post-recapture alterations."
}
```
```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document presents a consistent visual appearan

In [7]:
# V1 split: persona passed as system_instruction, task prompt as the user turn

image_list = list(zip(paths, images))  # image 1 and 2

for image_path, image in image_list:
    start = time.perf_counter()

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[task_prompt_v1, image],
        config=types.GenerateContentConfig(
            system_instruction=system_prompt_v1,
            temperature = 0.1
        ),
    )

    latency_s = round(time.perf_counter() - start, 2)

    print(response.text)

    log_result(
        batch_id=BATCH_ID,
        image_path=image_path,
        prompt_id="V1_split",
        model="gemini-2.5-flash",
        temperature=0.1,
        raw_response=response.text,
        latency_s=latency_s,
    )

```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document exhibits a consistent typography baseline across all text elements. Font weights, sizes, sharpness, and resolution are uniform within their respective categories (e.g., header text, body text, numerical values). Variations in font weight (e.g., bold for totals and base salary, regular for other items) appear to be a deliberate design choice and are applied consistently. There are no visual anomalies such as mismatched shadows, irregular edges, background discontinuities, or color/contrast shifts around specific text regions that would indicate post-recapture digital alteration. All text appears to be an integral part of the original digitally generated document.",
  "signals_detected": [],
  "notes": "The document's digital origin and clear rendering facilitate a confident assessment of authenticity based on visual signals."
}
```
```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reaso

## Review logged results

Load the full log (all batches) or just the current `BATCH_ID` to compare prompts/models without re-running anything.

In [10]:
df_all = load_results()
df_all[["timestamp", "batch_id", "image_path", "prompt_id", "model", "verdict", "confidence", "signal_types", "format", "latency_s"]]

,timestamp,batch_id,image_path,prompt_id,model,verdict,confidence,signal_types,format,latency_s
0,2026-06-11T02:20:36.332953+00:00,batch1,test_images/First_Tamper.png,V1,gemini-2.5-flash,authentic,high,[],True,9.42
1,2026-06-11T02:20:47.181659+00:00,batch1,test_images/first-tamper.png,V1,gemini-2.5-flash,tampered,medium,[resolution_mismatch],True,10.85
2,2026-06-11T02:20:53.951337+00:00,batch1,test_images/First_Tamper.png,V1_split,gemini-2.5-flash,authentic,high,[],True,6.76
3,2026-06-11T02:21:04.694144+00:00,batch1,test_images/first-tamper.png,V1_split,gemini-2.5-flash,authentic,high,[],True,10.74
4,2026-06-11T02:24:43.831825+00:00,batch1,test_images/First_Tamper.png,V1,gemini-2.5-flash,authentic,high,[],True,9.18
...,...,...,...,...,...,...,...,...,...,...
61,2026-06-11T06:29:43.637821+00:00,batch2,test_images/payslip10.jpeg,V1_split,gemini-2.5-flash,authentic,high,[],True,8.08
62,2026-06-11T06:29:51.011143+00:00,batch2,test_images/payslip2.png,V1_split,gemini-2.5-flash,authentic,high,[],True,7.37
63,2026-06-11T06:30:08.111892+00:00,batch2,test_images/payslip20.png,V1_split,gemini-2.5-flash,tampered,high,"[font_weight_inconsistency, resolution_mismatc...",True,17.10
64,2026-06-11T06:30:20.810830+00:00,batch2,test_images/payslip3.png,V1_split,gemini-2.5-flash,authentic,high,[],True,12.70


In [11]:
df_all.columns

Index(['timestamp', 'batch_id', 'image_path', 'prompt_id', 'model',
       'temperature', 'latency_s', 'raw_response', 'parsed_response', 'notes',
       'verdict', 'confidence', 'signal_types', 'format'],
      dtype='object')

In [ ]:
current_batch = load_results(batch_id=BATCH_ID)
current_batch[["image_path", "prompt_id", "verdict", "confidence", "signal_types", "format", "latency_s"]]

In [11]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      avg_logprobs=-1.2320455575918223,
      content=Content(
        parts=[
          Part(
            text="""```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document's typography, including font weight, sharpness, resolution, and color, is highly consistent across all text regions. Specifically, the numerical value '๖๐๐,๐๐๐' (600,000) matches the surrounding Thai text in every visual aspect. There are no signs of mismatched shadows, edges, backgrounds, or color/contrast discontinuities around any specific characters or numbers. All text appears to be rendered uniformly, suggesting it is part of the original digital generation rather than a post-recapture alteration.",
  "signals_detected": [],
  "notes": ""
}
```"""
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>
    ),
  ],
  create_time=datetime.dat